In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

INSURANCE = ["NLIC", "LICN", "NLICL", "HLI"]

def clean_number(value):
    if value is None:
        return None

    value = value.strip()

    if value in ["", "-", "N/A"]:
        return None

    value = value.replace(",", "")

    try:
        return float(value)
    except:
        return None

def scrape_company(symbol):
    url = f"https://merolagani.com/CompanyDetail.aspx?symbol={symbol}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Could not access {symbol}")
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        def get_value(label):
            for tag in soup.find_all(["td", "th"]):
                if label.lower() in tag.get_text(strip=True).lower():
                    sibling = tag.find_next_sibling(["td", "th"])
                    if sibling:
                        return sibling.get_text(strip=True)
            return None

        price = get_value("Market Price")
        pe = get_value("P/E Ratio")
        pb = get_value("PBV")

        company_data = {
            "Ticker": symbol,
            "Market Price": clean_number(price),
            "P/E Ratio": clean_number(pe),
            "P/B Ratio": clean_number(pb)
        }

        return company_data

    except Exception as e:
        print("Error:", e)
        return None

all_data = []

for company in INSURANCE:
    print("Scraping:", company)
    result = scrape_company(company)
    
    if result:
        all_data.append(result)
        
    time.sleep(1)

df = pd.DataFrame(all_data)

print("\nFinal Data:")
print(df)

df.to_csv("insurance_data.csv", index=False)

print("\nData saved to insurance_data.csv")

Scraping: NLIC
Scraping: LICN
Scraping: NLICL
Scraping: HLI

Final Data:
  Ticker  Market Price  P/E Ratio  P/B Ratio
0   NLIC         747.5     134.44       5.92
1   LICN         804.9     103.59       4.56
2  NLICL         567.1      58.10       4.35
3    HLI         328.9      86.33       3.27

Data saved to insurance_data.csv
